# Basic ML Model Deployment

## Import libraries

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.preprocessing import StandardScaler
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

## Fetch Data

In [15]:
#data=pd.read_csv('https://raw.githubusercontent.com/tkseneee/Dataset/master/Loan_data_ver2.csv')
data=pd.read_csv('Loan_data_ver2.csv')

## Explore Data

In [16]:
data.shape

(614, 6)

In [17]:
data.dtypes

Married             object
Education           object
ApplicantIncome      int64
LoanAmount         float64
Credit_History     float64
Loan_Status        float64
dtype: object

In [18]:
data.head(2)

,Married,Education,ApplicantIncome,LoanAmount,Credit_History,Loan_Status
0,No,Graduate,5849,NaN,1.0,0.10
1,Yes,Graduate,4583,128.0,1.0,0.32


In [19]:
# fetch features with missing values
data.isnull().sum()

Married             3
Education           0
ApplicantIncome     0
LoanAmount         22
Credit_History     50
Loan_Status         0
dtype: int64

3 features namely - Married,LoanAmount,Credit_History has missing values

In [20]:
data['Married'].value_counts()

Married
Yes    398
No     213
Name: count, dtype: int64

In [21]:
data['Education'].value_counts()

Education
Graduate        449
Not Graduate    127
HSC              38
Name: count, dtype: int64

In [22]:
# segreegating target & feature
X=data.drop('Loan_Status', axis=1)
y=data['Loan_Status']

In [23]:
# spliting data into train & validation set
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=48)

In [24]:
# fetching numeric features list
feat_num=list(X.select_dtypes(include=np.number).columns)


In [25]:
# fetching categorical features  list
feat_cat=list(X.select_dtypes(exclude=np.number).columns)

In [26]:
feat_cat

['Married', 'Education']

## Defining Data processing & Modeling  Pipeline

In [27]:
#  pipeline for numeric atures -missing values replacement using k-Nearest Neighbors follwed by StandardScaler() 
num_pipe=Pipeline([('imputer',KNNImputer()),('std_scale',StandardScaler())])



In [28]:
# pipeline for categorical faetures - missing category replacement by new category i.e. missing followed by one hot encoding 
feat_pipe = Pipeline([('imputer',SimpleImputer(strategy='constant', fill_value='Missing')), 
                      ('one_hot',(OneHotEncoder()))]) 



In [29]:
#combine data processing pipeline
data_pipeline=ColumnTransformer([('numeric',num_pipe,feat_num),
                                 ('categorical',feat_pipe, feat_cat)],
                                remainder='passthrough')



In [30]:
data_pipeline

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer', KNNImputer()),
                                                 ('std_scale',
                                                  StandardScaler())]),
                                 ['ApplicantIncome', 'LoanAmount',
                                  'Credit_History']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('one_hot', OneHotEncoder())]),
                                 ['Married', 'Education'])])

In [31]:
# adding ml-model into pipeline 
full_pipe=Pipeline([('pre_process',data_pipeline),('model',RandomForestRegressor())])

In [32]:
# training
full_pipe.fit(X_train,y_train)

Pipeline(steps=[('pre_process',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('std_scale',
                                                                   StandardScaler())]),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('one_hot',
                                                                   OneHotEncoder())]),
                                                  ['Married', 'Education'])])),
                ('model', RandomForestRegressor())])

In [33]:
# prediction
full_pipe.predict(X_test)

array([0.1401, 0.2396, 0.5274, 0.4353, 0.1565, 0.1345, 0.9567, 0.1966,
       0.2513, 0.2904, 0.2111, 0.9755, 0.2819, 0.1197, 0.0467, 0.0858,
       0.1144, 0.0445, 0.0754, 0.2698, 0.3314, 0.0677, 0.2731, 0.497 ,
       0.5952, 0.167 , 0.2876, 0.2248, 0.2249, 0.0229, 0.0996, 0.2297,
       0.3993, 0.9476, 0.1109, 0.2659, 0.2104, 0.6227, 0.9513, 0.1891,
       0.2899, 0.1418, 0.2904, 0.3423, 0.4044, 0.4738, 0.1071, 0.1693,
       0.2476, 0.2042, 0.1163, 0.4049, 0.0597, 0.1018, 0.9633, 0.0063,
       0.4321, 0.2806, 0.2749, 0.1907, 0.3093, 0.8815, 0.28  , 0.1093,
       0.4733, 0.221 , 0.077 , 0.2939, 0.1799, 0.2107, 0.3725, 0.1359,
       0.3179, 0.2612, 0.0709, 0.8142, 0.0928, 0.4609, 0.5717, 0.0407,
       0.2783, 0.1237, 0.3253, 0.7768, 0.4541, 0.1464, 0.1719, 0.0804,
       0.0613, 0.5146, 0.0574, 0.1961, 0.2464, 0.5629, 0.3713, 0.2053,
       0.9718, 0.0811, 0.0834, 0.4187, 0.4952, 0.0697, 0.0063, 0.0385,
       0.0927, 0.1112, 0.2965, 0.2101, 0.0545, 0.1219, 0.1133, 0.0794,
      

In [34]:
## can store numeric and categorical variables also as pickle file
pickle.dump(feat_num,open('feat_numv1','wb'))
pickle.dump(feat_cat,open('feat_catv1','wb'))

 

## Store the model as pickle file 

In [36]:
pickle.dump(full_pipe,open('full_pipeline','wb'))